In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_BASE_URL"]=os.getenv("BEDROCK_MANTLE_BASE_URL")
os.environ["OPENAI_API_KEY"]=os.getenv("BEDROCK_MANTLE_API_KEY")

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model("moonshotai.kimi-k2-thinking", model_provider="openai")

# Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

## Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The rating of the movie out of 10")

In [4]:
model_with_structure = model.with_structured_output(Movie)

model_with_structure

_ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-openai': '1.3.3'}}, output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x111b5fcb0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x111d9c830>, root_client=<openai.OpenAI object at 0x110eb6f90>, root_async_client=<openai.AsyncOpenAI object at 0x111d9c590>, model_name='moonshotai.kimi-k2-thinking', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_proxy=None, stream_chunk_timeout=120.0), kwargs={'response_format': <class '__main__.Movie'>, 'ls_structured_output_format': {'kwargs': {'method': 'json_schema', 'strict': None}, 'schema': {'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': '

In [5]:
model_with_structure.invoke("Provide the details about movie 3 Idiots")

Movie(title='3 Idiots', year=2009, director='Rajkumar Hirani', rating=8.4)

### Including RAW message

In [6]:
model_with_structure_raw = model.with_structured_output(Movie, include_raw=True)

model_with_structure_raw.invoke("Provide the details about movie 3 Idiots")

{'raw': AIMessage(content=' {\n  "title": "3 Idiots",\n  "year": 2009,\n  "director": "Rajkumar Hirani",\n  "rating": 8.4\n}', additional_kwargs={'parsed': Movie(title='3 Idiots', year=2009, director='Rajkumar Hirani', rating=8.4), 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 578, 'prompt_tokens': 38, 'total_tokens': 616, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 32}}, 'model_provider': 'openai', 'model_name': 'moonshotai.kimi-k2-thinking', 'system_fingerprint': None, 'id': 'chatcmpl-0de284a2-4dfb-4666-b1df-e9ad65fd9b7a', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f01f9-b3ec-7a73-a375-315fed46a453-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 578, 'total_tokens': 616, 'input_token_details': {'audio': 0, 'cache_read': 32}, 'output_token_details': {}}),
 'parsed': Movie(title='3 Idiots', year=2009, director='Rajkum

### Nested Fields

In [7]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str=Field("Name of the actor")
    role:str=Field("The role that was played by the actor")

class MovieDetail(BaseModel):
    title:str=Field("Title of the movie")
    year:int
    cast:list[Actor]
    generes:list[str]
    budget:float

model_with_structure_nested_field = model.with_structured_output(MovieDetail)
model_with_structure_nested_field.invoke("Provide the details about movie 3 Idiots")

MovieDetail(title='3 Idiots', year=2009, cast=[Actor(name='Aamir Khan', role='Ranchoddas "Rancho" Shamaldas Chanchad / Phunsukh Wangdu'), Actor(name='R. Madhavan', role='Farhan Qureshi'), Actor(name='Sharman Joshi', role='Raju Rastogi'), Actor(name='Kareena Kapoor', role='Pia Sahastrabuddhe'), Actor(name='Boman Irani', role='Dr. Viru Sahastrabuddhe (Virus)')], generes=['Comedy', 'Drama'], budget=55000000.0)

MovieDetail(title='3 Idiots', year=2009, cast=[Actor(name='Aamir Khan', role="Ranchoddas 'Rancho' Shamaldas Chanchad / Phunsukh Wangdu"), Actor(name='Kareena Kapoor Khan', role='Pia Sahastrabuddhe'), Actor(name='R. Madhavan', role='Farhan Qureshi'), Actor(name='Sharman Joshi', role='Raju Rastogi'), Actor(name='Boman Irani', role='Dr. Viru Sahastrabuddhe (Virus)'), Actor(name='Omi Vaidya', role="Chatur 'Silencer' Ramalingam")], generes=['Comedy', 'Drama'], budget=55000000.0)

## TypeDict

TypeDict provides a simple alternative using Python's build-in typing, ideal when you don't need runtime validation

In [8]:
from typing_extensions import TypedDict, Annotated

class MoveDict(TypedDict):
    title:Annotated[str,...,"The title of the movie"]
    year:Annotated[int,...,"The year the movie was released"]
    director:Annotated[str,...,"The director of the movie"]
    rating:Annotated[float,...,"The rating of the movie out of 10"]

model_with_typeddict = model.with_structured_output(MoveDict)

model_with_typeddict.invoke("Provide the details about movie 3 Idiots")

{'title': '3 Idiots',
 'year': 2009,
 'director': 'Rajkumar Hirani',
 'rating': 8.4}

### Nested Models

In [9]:
class ActorDict(TypedDict):
    name:str=Field("Name of the actor")
    role:str=Field("The role that was played by the actor")

class MovieDetailDict(TypedDict):
    title:str=Field("Title of the movie")
    year:int
    cast:list[ActorDict]
    generes:list[str]
    budget:float

model_with_detail_typedict = model.with_structured_output(MovieDetailDict)
model_with_detail_typedict.invoke("Provide the details about movie 3 Idiots")

{'title': '3 Idiots',
 'year': 2009,
 'cast': [{'name': 'Aamir Khan',
   'role': "Ranchoddas 'Rancho' Shamaldas Chanchad / Phunsukh Wangdu"},
  {'name': 'Kareena Kapoor', 'role': 'Pia Sahastrabuddhe'},
  {'name': 'R. Madhavan', 'role': 'Farhan Qureshi'},
  {'name': 'Sharman Joshi', 'role': 'Raju Rastogi'},
  {'name': 'Boman Irani', 'role': 'Dr. Viru Sahastrabuddhe (Virus)'},
  {'name': 'Omi Vaidya', 'role': "Chatur 'Silencer' Ramalingam"}],
 'generes': ['comedy', 'drama'],
 'budget': 55000000}

## DataClasses

A data class is a class typically containing mainly data, although there aren't really any restrictions. You create is using @dataclass decorator

In [10]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_BASE_URL"]=os.getenv("BEDROCK_MANTLE_BASE_URL")
os.environ["OPENAI_API_KEY"]=os.getenv("BEDROCK_MANTLE_API_KEY")

In [11]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="moonshotai.kimi-k2-thinking",
    base_url=os.environ["OPENAI_BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
    temperature=0,
)

### Pydantic

In [13]:
from langchain.agents import create_agent
from pydantic import BaseModel, Field

class ContactInfo(BaseModel):
    """Contact information of a person"""
    name:str=Field(description="Name of the person")
    email:str=Field(description="Email of the person")
    phone:str=Field(description="Phone of the person")

agent = create_agent(
    model=model,
    tools=[],
    response_format=ContactInfo
)

response = agent.invoke({
    "messages": [{"role":"user", "content": "Extract contact info from: John Doe, JohnDoe@example.com, (555) 123-4567"}]
})

print(response["structured_response"])

name='John Doe' email='JohnDoe@example.com' phone='(555) 123-4567'


### TypeDict

In [ ]:
from langchain.agents import create_agent
from typing_extensions import Annotated, TypedDict

class ContactInfoDict(TypedDict):
    """Contact information of a person"""
    name:Annotated[str,...,"Name of the person"]
    email:Annotated[str,...,"Email of the person"]
    phone:Annotated[str,...,"Phone of the person"]

agent = create_agent(
    model=model,
    tools=[],
    response_format=ContactInfoDict
)

response = agent.invoke({
    "messages": [{"role":"user", "content": "Extract contact info from: John Doe, JohnDoe@example.com, (555) 123-4567"}]
})

print(response["structured_response"])

{'name': 'John Doe', 'email': 'JohnDoe@example.com', 'phone': '(555) 123-4567'}


In [ ]:
from langchain.agents import create_agent
from dataclasses import dataclass

class ContactInfoDataClass:
    """Contact information of a person"""
    name:str # Name of the person
    email:str # Email of the person
    phone:str # Phone number of the person

agent = create_agent(
    model=model,
    tools=[],
    response_format=ContactInfoDataClass
)

response = agent.invoke({
    "messages": [{"role":"user", "content": "Extract contact info from: John Doe, JohnDoe@example.com, (555) 123-4567"}]
})

print(response["structured_response"])